# 3막. 모델링 - 지도학습 + Isolation Forest

이전 두 노트북에서 확인한 내용:

- [1막(`01_eda.ipynb`)](./01_eda.ipynb): 사기 비율 0.58%의 극단적 불균형, 사기 거래 금액이 정상보다 평균 8배 큼.
- [2막(`02_clustering.ipynb`)](./02_clustering.ipynb): k=4 클러스터링 결과 클러스터별 사기 비율이 0.155%~0.958%까지 벌어졌지만, feature 17차원 중 14차원이 category 원-핫이라 클러스터가 category에 구조적으로 지배됐을 가능성이 있다는 한계를 명시함.

이 노트북의 목적은 두 가지다.

1. **지도학습 모델과 비지도학습(Isolation Forest)을 공정하게 비교**한다. 여기서 "공정하게"란 동일한 시간 기준 train/test 분할, 동일한 feature 집합을 사용한다는 뜻이다.
2. **2막에서 만든 `cluster`/`cluster_dist` feature가 실제로 예측 성능에 기여하는지 검증**한다. 2막의 한계(category 지배)가 실제로 이 feature를 쓸모없게 만들었는지, 아니면 그럼에도 신호가 남아있는지 데이터로 확인한다.

최종적으로 채택한 모델의 예측 확률은 4막(`04_cost_optimization.ipynb`)에서 threshold 최적화에 쓰인다.

## Section 0. 환경설정 + train/test 분할

In [1]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

# 시스템에 설치된 한글 폰트를 우선순위대로 탐색해 적용 (없으면 기본 폰트 유지)
# 주의: sns.set_style()이 rcParams의 font.family를 초기화하므로 반드시 그 다음에 설정해야 한다.
korean_font_candidates = [
    "Noto Sans CJK KR",
    "AppleGothic",
    "Apple SD Gothic Neo",
    "NanumGothic",
    "Malgun Gothic",
]
available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in korean_font_candidates:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break
else:
    warnings.warn("한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")

plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", None)
np.random.seed(42)
print(f"적용된 폰트: {plt.rcParams['font.family']}")

적용된 폰트: ['AppleGothic']


**왜 랜덤 분할이 아니라 시간 기준 분할인가**

실제 카드사의 FDS는 과거 거래로 학습한 모델로 **미래에 들어올** 거래를 예측하는 구조다. 랜덤 분할은 미래 시점의 거래가 학습 데이터에 섞여 들어가는 것을 허용하므로, 모델이 실제로는 알 수 없었을 미래 정보(예: 특정 시기에 유행한 사기 패턴)를 미리 학습해 성능이 실제보다 과대평가된다.

따라서 `trans_date_trans_time` 기준으로 전체 데이터를 정렬한 뒤, **가장 오래된 80%를 train, 가장 최근 20%를 test**로 분리한다. 이렇게 하면 test 성능이 "이 모델을 그 시점에 실제로 배포했다면 어땠을까"에 더 가까운 근사치가 된다.

In [2]:
df = pd.read_csv("../data/credit.csv")
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df = df.sort_values("trans_date_trans_time").reset_index(drop=True)

print(f"shape: {df.shape}")
print(f"기간: {df['trans_date_trans_time'].min()} ~ {df['trans_date_trans_time'].max()}")

shape: (1296675, 24)
기간: 2019-01-01 00:00:18 ~ 2020-06-21 12:13:37


In [3]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """카드 소지자-가맹점 간 거리(km). 2막(02_clustering)과 동일한 정의."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


df["amt_log"] = np.log1p(df["amt"])
df["trans_hour"] = df["trans_date_trans_time"].dt.hour
df["distance_km"] = haversine_distance(df["lat"], df["long"], df["merch_lat"], df["merch_long"])

category_dummies = pd.get_dummies(df["category"], prefix="cat")
df = pd.concat([df, category_dummies], axis=1)

numeric_cols = ["amt_log", "trans_hour", "distance_km"]
category_cols = category_dummies.columns.tolist()
base_feature_cols = numeric_cols + category_cols

print(f"base feature 개수: {len(base_feature_cols)}")

base feature 개수: 17

**cluster feature의 lookahead(데이터 누수) 문제와 수정**

**문제 진단**: 2막(`02_clustering.ipynb`)의 K-means는 train/test 분할 없이 **전체 1,296,675건**(`X_scaled`, df 전체)에 대해 `KMeans(...).fit_predict(X_scaled)`를 실행했다. 즉 `data/cluster_features.csv`에 저장된 `cluster`, `cluster_dist` 값에는 test 구간(미래) 거래들의 분포 정보가 클러스터 중심(centroid) 계산에 이미 반영되어 있다 — 실제 배포 시점에는 알 수 없었어야 할 정보다. 이 노트북의 이전 버전은 이 파일을 그대로 merge해서 사용했고, 그 결과 Section 3에서 관찰한 cluster feature의 PR-AUC 기여도(+0.0104)에는 이 lookahead가 섞여 있었다.

**조치**: `data/cluster_features.csv`를 더 이상 사용하지 않는다. 대신 아래에서 먼저 시간 기준으로 train/test를 분리한 뒤, base feature(`amt_log`, `trans_hour`, `distance_km`, category 원-핫)로 `StandardScaler`와 `KMeans(k=4)`를 **train 데이터에만 `fit`**하고, test 데이터는 그 학습된 scaler·KMeans로 `transform`/`predict`만 수행해 `cluster`, `cluster_dist`를 만든다. k=4는 2막에서 elbow/실루엣 근거로 선택한 값을 그대로 따른다. 이렇게 하면 test 데이터의 어떤 정보도 클러스터 중심 계산에 관여하지 않는다.

**결과가 어떻게 바뀌었는지**

| | 수정 전 (lookahead 있음) | 수정 후 (train만 fit) |
|---|---|---|
| Section 3 — cluster 미포함 PR-AUC | 0.8300 | 0.8300 (동일, cluster를 안 쓰므로 당연) |
| Section 3 — cluster 포함 PR-AUC | 0.8404 | 0.8400 |
| Section 3 — PR-AUC 개선폭 | **+0.0104** | **+0.0100** |
| Section 4 — IF 이상치 중 사기 비율 | 6.90% | **17.84%** |
| Section 4 — IF recall (사기 검출률) | 7.0% | **18.1%** |
| Section 4 — 지도학습이 놓친 건 중 IF가 회수 | 3.7% | **10.6%** |

두 가지 결과가 나왔는데, 예상과 반대다.

1. **Section 3(지도학습 feature 기여도)은 거의 안 바뀌었다.** PR-AUC 개선폭이 +0.0104 → +0.0100으로, 차이는 0.0004에 불과하다. 즉 "lookahead 때문에 cluster feature의 기여도가 과대평가됐을 것"이라는 앞선 우려는 이 경우엔 기우였다 — XGBoost 입장에서는 cluster 정보 대부분이 어차피 category/hour와 중복이라, lookahead가 있든 없든 실질적으로 비슷한 정보량을 준 것으로 보인다.
2. **Section 4(Isolation Forest)는 오히려 크게 개선됐다.** 이상치 중 사기 비율이 6.90% → 17.84%로, recall은 7.0% → 18.1%로 두 배 이상 올랐다. 이유를 추정하면: lookahead가 있던 버전에서는 test 포인트 자신이 클러스터 중심 계산에 참여했기 때문에 `cluster_dist`(중심까지 거리)가 인위적으로 작게 나와 test의 이상치들이 "덜 이상해 보이게" 눌려 있었다. train만으로 중심을 고정하면, 실제 배포 환경처럼 학습 시점에 보지 못한 최근 패턴이 중심에서 진짜로 멀어져 이상치로 더 잘 잡힌다. 결과적으로 **lookahead는 Isolation Forest의 실제 성능을 과소평가**하고 있었다.

In [4]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print(f"train: {train_df.shape[0]:,}건 ({train_df['trans_date_trans_time'].min()} ~ {train_df['trans_date_trans_time'].max()})")
print(f"test:  {test_df.shape[0]:,}건 ({test_df['trans_date_trans_time'].min()} ~ {test_df['trans_date_trans_time'].max()})")
print(f"train 사기 비율: {train_df['is_fraud'].mean()*100:.3f}%")
print(f"test  사기 비율: {test_df['is_fraud'].mean()*100:.3f}%")

train: 1,037,340건 (2019-01-01 00:00:18 ~ 2020-03-06 07:15:17)
test:  259,335건 (2020-03-06 07:16:43 ~ 2020-06-21 12:13:37)
train 사기 비율: 0.575%
test  사기 비율: 0.593%


In [5]:
cluster_k = 4  # 2막(02_clustering)에서 elbow/실루엣 근거로 선택한 k와 동일

cluster_scaler = StandardScaler()
X_train_cluster = cluster_scaler.fit_transform(train_df[base_feature_cols])
X_test_cluster = cluster_scaler.transform(test_df[base_feature_cols])

kmeans_train_only = KMeans(n_clusters=cluster_k, n_init=10, random_state=42)
train_df["cluster"] = kmeans_train_only.fit_predict(X_train_cluster)  # train에만 fit
test_df["cluster"] = kmeans_train_only.predict(X_test_cluster)  # test는 predict만

train_df["cluster_dist"] = np.linalg.norm(
    X_train_cluster - kmeans_train_only.cluster_centers_[train_df["cluster"].values], axis=1
)
test_df["cluster_dist"] = np.linalg.norm(
    X_test_cluster - kmeans_train_only.cluster_centers_[test_df["cluster"].values], axis=1
)

for part_df in (train_df, test_df):
    for c in range(cluster_k):
        part_df[f"clus_{c}"] = (part_df["cluster"] == c).astype(int)

clus_cols = [f"clus_{c}" for c in range(cluster_k)]
cluster_feature_cols = clus_cols + ["cluster_dist"]
full_feature_cols = base_feature_cols + cluster_feature_cols

print(f"train cluster 분포: {train_df['cluster'].value_counts().sort_index().to_dict()}")
print(f"test  cluster 분포: {test_df['cluster'].value_counts().sort_index().to_dict()}")
print(f"전체 feature 개수 (base + cluster): {len(full_feature_cols)}")

train cluster 분포: {0: 174327, 1: 93330, 2: 290031, 3: 479652}
test  cluster 분포: {0: 43378, 1: 23342, 2: 72885, 3: 119730}
전체 feature 개수 (base + cluster): 22


In [6]:
def make_xy(feature_cols):
    """feature_cols로 train/test 행렬을 만들고, train에만 fit한 StandardScaler로 둘 다 변환한다."""
    X_train = train_df[feature_cols].astype(float)
    X_test = test_df[feature_cols].astype(float)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    y_train = train_df["is_fraud"].values
    y_test = test_df["is_fraud"].values
    return X_train_scaled, X_test_scaled, y_train, y_test


def evaluate(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, y_proba),
    }


def train_eval(model_type, balanced, feature_cols):
    """model_type: 'LogisticRegression' 또는 'XGBoost'. balanced=True면 클래스 불균형을 보정한다."""
    X_train, X_test, y_train, y_test = make_xy(feature_cols)

    if model_type == "LogisticRegression":
        model = LogisticRegression(
            max_iter=1000, class_weight="balanced" if balanced else None, random_state=42
        )
    elif model_type == "XGBoost":
        neg, pos = np.bincount(y_train)
        model = XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1,
            scale_pos_weight=(neg / pos) if balanced else 1,
        )
    else:
        raise ValueError(model_type)

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    metrics = evaluate(y_test, pred, proba)
    return model, metrics, pred, proba

## Section 1. Baseline 모델

`class_weight`/`scale_pos_weight` 같은 불균형 보정 없이, base feature(amt_log, trans_hour, distance_km, category 원-핫)만으로 LogisticRegression과 XGBoost를 학습한다.

In [7]:
_, lr_base_metrics, _, _ = train_eval("LogisticRegression", balanced=False, feature_cols=base_feature_cols)
_, xgb_base_metrics, _, _ = train_eval("XGBoost", balanced=False, feature_cols=base_feature_cols)

baseline_results = {"LogisticRegression": lr_base_metrics, "XGBoost": xgb_base_metrics}
pd.DataFrame(baseline_results).T.round(4)

,accuracy,precision,recall,f1,pr_auc
LogisticRegression,0.9939,0.1370,0.0065,0.0124,0.3030
XGBoost,0.9973,0.8325,0.6853,0.7518,0.8071


**Accuracy가 오해를 주는 이유**

| model | accuracy | precision | recall | f1 | pr_auc |
|---|---|---|---|---|---|
| LogisticRegression | 0.9939 | 0.1370 | 0.0065 | 0.0124 | 0.3030 |
| XGBoost | 0.9973 | 0.8325 | 0.6853 | 0.7518 | 0.8071 |

LogisticRegression baseline이 정확히 이 문제를 보여준다. accuracy는 99.39%로 매우 높아 보이지만, recall은 0.65%에 불과하다 — 실제 사기 1,538건 중 겨우 10건 안팎만 잡아냈다는 뜻이다. 1막에서 확인한 test 사기 비율이 0.593%였으니, **"모든 거래를 정상으로만 예측"해도 정확도는 약 99.41%**가 나온다. 즉 이 모델의 99.39% accuracy는 사실상 아무것도 하지 않는 것과 거의 같은 점수다.

반면 XGBoost baseline은 accuracy(99.73%)와 recall(68.53%)이 함께 높아 언뜻 accuracy도 쓸만해 보이지만, 이는 XGBoost가 우연히 두 지표에서 모두 잘한 것일 뿐 accuracy 자체가 신뢰할 지표라는 뜻은 아니다. LogisticRegression의 사례처럼 accuracy만으로는 "사기를 거의 못 잡는 모델"과 "사기를 잘 잡는 모델"을 구분할 수 없으므로, 이후 모델 비교는 PR-AUC를 기준으로 한다.

## Section 2. 불균형 처리 후 모델

LogisticRegression은 `class_weight="balanced"`, XGBoost는 그에 대응하는 `scale_pos_weight`(= train 정상건수/사기건수 비율)로 불균형을 보정한다. 두 방식 모두 소수 클래스(사기)의 오분류에 더 큰 페널티를 부여한다는 점에서 같은 아이디어다.

In [8]:
_, lr_bal_metrics, _, _ = train_eval("LogisticRegression", balanced=True, feature_cols=base_feature_cols)
_, xgb_bal_metrics, _, _ = train_eval("XGBoost", balanced=True, feature_cols=base_feature_cols)

balanced_results = {"LogisticRegression": lr_bal_metrics, "XGBoost": xgb_bal_metrics}

comparison = pd.concat(
    [
        pd.DataFrame(baseline_results).T.assign(setting="baseline"),
        pd.DataFrame(balanced_results).T.assign(setting="balanced"),
    ]
).round(4)
comparison = comparison.reset_index().rename(columns={"index": "model"}).set_index(["model", "setting"])
comparison

,,accuracy,precision,recall,f1,pr_auc
model,setting,,,,,
LogisticRegression,baseline,0.9939,0.1370,0.0065,0.0124,0.3030
XGBoost,baseline,0.9973,0.8325,0.6853,0.7518,0.8071
LogisticRegression,balanced,0.7484,0.0172,0.7367,0.0336,0.1419
XGBoost,balanced,0.9837,0.2597,0.9467,0.4076,0.8300


In [9]:
all_results = {
    ("LogisticRegression", "baseline"): lr_base_metrics,
    ("XGBoost", "baseline"): xgb_base_metrics,
    ("LogisticRegression", "balanced"): lr_bal_metrics,
    ("XGBoost", "balanced"): xgb_bal_metrics,
}
best_model_type, best_setting = max(all_results, key=lambda k: all_results[k]["pr_auc"])
best_balanced = best_setting == "balanced"

print(f"PR-AUC 기준 최고 성능: {best_model_type} ({best_setting}), PR-AUC={all_results[(best_model_type, best_setting)]['pr_auc']:.4f}")

PR-AUC 기준 최고 성능: XGBoost (balanced), PR-AUC=0.8300


**baseline 대비 balanced 개선 여부**

| model | setting | pr_auc | recall |
|---|---|---|---|
| LogisticRegression | baseline | 0.3030 | 0.0065 |
| LogisticRegression | balanced | 0.1419 | 0.7367 |
| XGBoost | baseline | 0.8071 | 0.6853 |
| XGBoost | balanced | 0.8300 | 0.9467 |

두 모델이 반대 방향으로 반응했다. **XGBoost는 balanced 적용으로 PR-AUC가 0.8071 → 0.8300으로 개선**됐고 recall도 68.5% → 94.7%로 크게 올라갔다. 반면 **LogisticRegression은 balanced 적용으로 PR-AUC가 오히려 0.3030 → 0.1419로 악화**됐다 — recall은 0.65% → 73.67%로 극적으로 올랐지만, precision이 1.7%까지 떨어지며 전체적인 순위 매기기 품질(PR-AUC는 threshold에 무관하게 모든 지점에서의 precision-recall 트레이드오프를 종합한 지표)은 나빠졌다.

즉 `class_weight="balanced"`가 항상 PR-AUC를 개선하는 것은 아니다. 결정 threshold 근처의 recall/precision 균형은 바꾸지만, 모델이 사기와 정상을 얼마나 잘 "순위 매기는지" 자체는 모델 구조(선형 vs 트리 앙상블)에 더 좌우된다. PR-AUC 기준 최종 선택은 **XGBoost (balanced), PR-AUC=0.8300**이다.

## Section 3. Feature 기여도 확인 — 핵심

Section 2에서 PR-AUC가 가장 높았던 모델·설정(`best_model_type`, `best_setting`)을 그대로 사용해, `cluster`/`cluster_dist` feature를 뺀 모델과 넣은 모델의 PR-AUC를 비교한다. 2막에서 만든 클러스터링 결과가 실제로 예측에 기여하는지 확인하는 이 노트북의 핵심 실험이다.

In [10]:
model_without_cluster, metrics_without_cluster, pred_without_cluster, proba_without_cluster = train_eval(
    best_model_type, balanced=best_balanced, feature_cols=base_feature_cols
)
model_with_cluster, metrics_with_cluster, pred_with_cluster, proba_with_cluster = train_eval(
    best_model_type, balanced=best_balanced, feature_cols=full_feature_cols
)

ablation_df = pd.DataFrame(
    {"without_cluster_feature": metrics_without_cluster, "with_cluster_feature": metrics_with_cluster}
).T.round(4)
ablation_df["pr_auc_delta"] = (
    ablation_df["pr_auc"] - ablation_df.loc["without_cluster_feature", "pr_auc"]
).round(4)
ablation_df

,accuracy,precision,recall,f1,pr_auc,pr_auc_delta
without_cluster_feature,0.9837,0.2597,0.9467,0.4076,0.83,0.00
with_cluster_feature,0.9847,0.2717,0.9447,0.4220,0.84,0.01


**클러스터링 feature가 실제로 기여했는가**

*(위 lookahead 수정 이후 수치. train 데이터에만 fit한 K-means 기준)*

| | pr_auc | recall | precision |
|---|---|---|---|
| cluster 미포함 | 0.8300 | 0.9467 | 0.2597 |
| cluster 포함 | 0.8400 | 0.9447 | 0.2717 |

`cluster`/`cluster_dist`를 추가하자 PR-AUC가 0.8300 → 0.8400으로 **+0.0100(약 1.2% 상대 개선)** 올랐다. recall은 94.67% → 94.47%로 미세하게 낮아졌고 precision은 25.97% → 27.17%로 개선됐다 — 대체로 사기 검출력은 유지한 채 오탐(FP)을 줄이는 방향으로 기여했다.

기여했지만 **크지 않은 이유**는 2막에서 명시한 한계와 맞닿아 있다. `cluster`는 category(14차원 원-핫)와 `trans_hour`가 이미 base feature에 그대로 들어가 있는 상태에서 만들어진 값이라, 그 정보의 상당 부분이 base feature와 중복(redundant)된다. XGBoost 같은 트리 모델은 이미 category·hour 조합을 스스로 학습할 수 있어서, cluster label이 주는 추가 정보는 "category+hour의 비선형 조합을 미리 요약해준 값" 정도의 한정된 가치만 갖는다. `cluster_dist`(클러스터 중심까지의 거리, 이상치 정도)가 그나마 base feature에 없던 새로운 신호를 보탰을 가능성이 높다.

**lookahead 검증 결과**: 위에서 train 데이터에만 K-means를 다시 fit해 재검증한 결과, PR-AUC 개선폭은 +0.0104(수정 전) → +0.0100(수정 후)으로 거의 변하지 않았다. 즉 이 +0.01 수준의 기여도는 lookahead 때문에 부풀려진 것이 아니라 실제 신호였다는 뜻이다. 다만 Section 4에서 보듯 Isolation Forest 쪽에는 lookahead가 훨씬 큰 영향을 미쳤다 — feature 하나의 lookahead가 모델마다 다르게 작용할 수 있다는 것을 보여주는 사례다.

## Section 4. Isolation Forest 비교

`is_fraud` 라벨을 전혀 사용하지 않고 Isolation Forest를 학습한 뒤, 이상치로 분류한 거래 중 실제 사기 비율을 사후 검증한다. `contamination`은 1막에서 확인한 실제 사기 비율(0.58%)의 근사치로 설정한다. Section 3에서 PR-AUC가 더 높았던 feature 집합(cluster 포함 여부)을 그대로 사용해, 지도학습 모델과 동일한 조건에서 비교한다.

In [11]:
use_cluster_features = metrics_with_cluster["pr_auc"] >= metrics_without_cluster["pr_auc"]
final_feature_cols = full_feature_cols if use_cluster_features else base_feature_cols
final_model = model_with_cluster if use_cluster_features else model_without_cluster
final_pred = pred_with_cluster if use_cluster_features else pred_without_cluster
final_proba = proba_with_cluster if use_cluster_features else proba_without_cluster
final_metrics = metrics_with_cluster if use_cluster_features else metrics_without_cluster

X_train_final, X_test_final, y_train_final, y_test_final = make_xy(final_feature_cols)

contamination = 0.0058  # 1막에서 확인한 실제 사기 비율(0.58%) 근사치

iso_forest = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
iso_forest.fit(X_train_final)  # is_fraud 라벨 사용하지 않음

iso_pred_test = iso_forest.predict(X_test_final)  # 1: 정상, -1: 이상치
iso_anomaly_mask = iso_pred_test == -1

anomaly_fraud_rate = y_test_final[iso_anomaly_mask].mean() * 100
overall_test_fraud_rate = y_test_final.mean() * 100

print(f"최종 feature 집합: {'cluster 포함' if use_cluster_features else 'cluster 미포함'} ({len(final_feature_cols)}개)")
print(f"이상치로 분류된 거래: {iso_anomaly_mask.sum():,}건 ({iso_anomaly_mask.mean()*100:.3f}% of test)")
print(f"이상치 중 실제 사기 비율: {anomaly_fraud_rate:.2f}%")
print(f"test set 전체 사기 비율: {overall_test_fraud_rate:.3f}%")
print(f"→ Isolation Forest가 찍은 이상치는 전체 평균 대비 {anomaly_fraud_rate/overall_test_fraud_rate:.1f}배 밀도로 사기를 포함")

최종 feature 집합: cluster 포함 (22개)
이상치로 분류된 거래: 1,564건 (0.603% of test)
이상치 중 실제 사기 비율: 17.84%
test set 전체 사기 비율: 0.593%
→ Isolation Forest가 찍은 이상치는 전체 평균 대비 30.1배 밀도로 사기를 포함


In [12]:
actual_fraud_idx = set(np.where(y_test_final == 1)[0])
supervised_caught = set(np.where((final_pred == 1) & (y_test_final == 1))[0])
iso_caught = set(np.where(iso_anomaly_mask & (y_test_final == 1))[0])

n_actual_fraud = len(actual_fraud_idx)
n_supervised_caught = len(supervised_caught)
n_iso_caught = len(iso_caught)
n_overlap = len(supervised_caught & iso_caught)

supervised_missed = actual_fraud_idx - supervised_caught
n_supervised_missed = len(supervised_missed)
n_recovered_by_iso = len(supervised_missed & iso_caught)

print(f"실제 사기 건수 (test): {n_actual_fraud:,}")
print(f"지도학습({best_model_type})이 잡은 사기: {n_supervised_caught:,} ({n_supervised_caught/n_actual_fraud*100:.1f}%)")
print(f"Isolation Forest가 잡은 사기: {n_iso_caught:,} ({n_iso_caught/n_actual_fraud*100:.1f}%)")
print(f"두 방법 모두 잡은 사기(겹침): {n_overlap:,} (지도학습이 잡은 것 중 {n_overlap/n_supervised_caught*100:.1f}%)")
print(f"지도학습이 놓친 사기: {n_supervised_missed:,}")
print(f"  → 그 중 Isolation Forest가 잡아낸 비율: {n_recovered_by_iso:,} / {n_supervised_missed:,} ({n_recovered_by_iso/n_supervised_missed*100:.1f}%)")

실제 사기 건수 (test): 1,538
지도학습(XGBoost)이 잡은 사기: 1,453 (94.5%)
Isolation Forest가 잡은 사기: 279 (18.1%)
두 방법 모두 잡은 사기(겹침): 270 (지도학습이 잡은 것 중 18.6%)
지도학습이 놓친 사기: 85
  → 그 중 Isolation Forest가 잡아낸 비율: 9 / 85 (10.6%)


**Isolation Forest 해석**

*(lookahead 수정 이후 수치. cluster_dist도 train-only K-means 기준으로 재계산됨)*

- 이상치로 분류된 거래: 1,564건 (test의 0.603%, 목표 contamination 0.58%와 거의 일치)
- 그 중 실제 사기 비율: **17.84%** — test 전체 사기 비율(0.593%)의 약 **30.1배** 밀도. lookahead가 있던 버전(6.90%, 11.6배)보다 훨씬 뚜렷한 신호다.
- 지도학습(XGBoost, balanced, cluster 포함)과 비교: 실제 사기 1,538건 중 **지도학습은 1,453건(94.5%)**을 잡았고, **Isolation Forest는 279건(18.1%)**을 잡았다 — 여전히 큰 격차지만 수정 전(7.0%)보다 두 배 이상 개선됐다.
- 두 방법이 함께 잡은 건 270건으로, 지도학습이 잡은 것의 18.6%.
- 지도학습이 놓친 85건 중 Isolation Forest가 대신 잡아낸 건은 **9건(10.6%)**이다. 수정 전(3.7%)보다 늘었지만, 여전히 지도학습을 대체하기보다는 보조적인 역할에 그친다.

`cluster_dist`를 train 데이터에만 fit한 K-means로 다시 계산하자 Isolation Forest 성능이 크게 좋아진 것은, 이 feature가 "학습 시점에 보지 못한 낯선 패턴"을 더 정직하게 반영하게 됐기 때문으로 보인다. 그럼에도 여전히 지도학습 대비 recall이 5배 이상 낮으므로, Isolation Forest를 단독 채택하거나 최종 모델의 핵심 축으로 쓰기엔 부족하다.

## 결론

1. 최종 채택 모델은 **XGBoost(`scale_pos_weight` 균형 보정) + cluster feature 포함**이다(test PR-AUC=0.8400, recall=94.5%). cluster feature는 train 데이터에만 K-means를 fit해 lookahead를 제거한 뒤에도 PR-AUC를 0.8300→0.8400으로 개선시켰다(+0.0100) — lookahead가 있던 버전(+0.0104)과 거의 같은 크기라, 이 개선은 실제 신호였다고 판단한다.
2. Isolation Forest는 lookahead 제거 후 recall이 7.0%→18.1%로 뚜렷이 개선됐지만(이상치 중 사기 비율 17.84%, 전체 대비 30.1배), 지도학습(94.5%)과는 여전히 큰 격차가 있어 단독 채택이나 지도학습 대체용으로는 부적합하다(지도학습이 놓친 건 중 10.6%만 회수).
3. 최종 모델의 test set 예측 확률(`predicted_proba`)과 Isolation Forest 이상치 점수를 `data/model_predictions.csv`에 저장했다. 4막(`04_cost_optimization.ipynb`)은 이 `predicted_proba`를 기준으로 FN/FP 비용을 반영한 threshold를 탐색한다.

In [13]:
iso_anomaly_score = -iso_forest.score_samples(X_test_final)  # 클수록 이상치에 가까움

model_predictions = test_df[["trans_num", "trans_date_trans_time", "amt", "is_fraud"]].copy()
model_predictions["predicted_proba"] = final_proba
model_predictions["iso_forest_anomaly_score"] = iso_anomaly_score
model_predictions["iso_forest_is_anomaly"] = iso_anomaly_mask.astype(int)

model_predictions.to_csv("../data/model_predictions.csv", index=False)

print(f"저장 완료: data/model_predictions.csv ({len(model_predictions):,} rows)")
print(f"최종 모델: {best_model_type} ({best_setting}), {'cluster 포함' if use_cluster_features else 'cluster 미포함'}")
model_predictions.head()

저장 완료: data/model_predictions.csv (259,335 rows)
최종 모델: XGBoost (balanced), cluster 포함


,trans_num,trans_date_trans_time,amt,is_fraud,predicted_proba,iso_forest_anomaly_score,iso_forest_is_anomaly
1037340,20fbf26eb491ec4173d969be75cf6184,2020-03-06 07:16:43,45.16,0,1.134023e-05,0.453136,0
1037341,8e949d5f9bdb82f28734446b6e9a0cd0,2020-03-06 07:18:00,185.51,0,3.675040e-04,0.453476,0
1037342,4e4cda984382d10201724de06c346ea0,2020-03-06 07:19:45,122.96,0,5.327044e-06,0.511733,0
1037343,39abb5b0b9a3bf23980c437dc1ccb070,2020-03-06 07:21:19,1.89,0,1.703667e-05,0.484845,0
1037344,318635f145322c5e9c96af62205b84d7,2020-03-06 07:21:23,5.05,0,3.412703e-07,0.470467,0
